# Personality Atlas Embedding Projector

Interactive visualization of 6,694 factor chain embeddings across the 44-model atlas. Each of the 358 factors is encoded as a **factor chain**, a structured tuple of trait adjectives, synonyms, behavioral verbs, and associated nouns. Every factor chain row is then embedded as a 1,536-dimensional vector, creating a single searchable space where constructs from all 44 models can be compared and clustered.

**Data source:** GitHub `Wildertrek/survey/Embeddings`
**Embeddings:** OpenAI text-embedding-3-small (1536-dim)
**Models:** OCEAN, HEXACO, NPI, BDI, GAD-7, MCMI, and 38 others

Run this notebook in Google Colab with no setup required (all dependencies auto-installed).

## Section 1: Setup & Install

In [ ]:
!pip install -q plotly scikit-learn umap-learn datasets huggingface_hub numpy pandas

## Section 2: Load Embeddings from GitHub

In [ ]:
import numpy as np
import pandas as pd
import requests
import io
import json

print("Loading personality atlas (1536-dim embeddings from GitHub)...")

MODELS_URL = "https://raw.githubusercontent.com/Wildertrek/survey/main"
embeddings_base = f"{MODELS_URL}/Embeddings"

all_embeddings = []
all_metadata = []
model_indices = {}

for model_dir in sorted(['aam', 'bdi', 'bt', 'cest', 'cmoa', 'cs', 'disc', 'dt4', 'dtm', 'em', 'epm', 
                         'ffni', 'ffni_sf', 'fsls', 'ftm', 'gad7', 'hex', 'hsns', 'ipn', 'mbti', 
                         'mcmi', 'mcmin', 'mmpi', 'mst', 'narq', 'npi', 'ocean', 'papc', 'pct', 
                         'pni', 'rft', 'riasec', 'rit', 'scid', 'scm', 'sdt', 'sixteenpf', 'stbv', 
                         'tat', 'tci', 'tei', 'tki', 'tmp', 'wais']):
    try:
        emb_url = f"{embeddings_base}/{model_dir}_embeddings.csv"
        response = requests.get(emb_url, timeout=10)
        if response.status_code == 200:
            emb_df = pd.read_csv(io.StringIO(response.text))
            embeddings = [json.loads(e) for e in emb_df['Embedding']]
            embeddings_array = np.array(embeddings, dtype=np.float32)
            all_embeddings.append(embeddings_array)
            
            model_name = model_dir.upper()
            for _, row in emb_df.iterrows():
                all_metadata.append({
                    'model': model_name,
                    'factor': str(row.get('Factor', '')),
                    'adjective': str(row.get('Adjective', '')),
                })
            
            model_indices[model_name] = (len(all_embeddings) - 1, len(embeddings_array))
            print(f"  {model_name:15s}: {len(embeddings):4d} x {len(embeddings[0])}-dim")
    except Exception as e:
        print(f"  {model_dir.upper()}: {str(e)[:50]}")

print(f"\nLoaded {len(all_embeddings)} models")

X = np.vstack(all_embeddings)
print(f"Combined shape: {X.shape}")

CATEGORY_MAP = {
    "OCEAN": "Trait-Based", "MBTI": "Trait-Based", "HEX": "Trait-Based",
    "EPM": "Trait-Based", "SIXTEENPF": "Trait-Based", "FTM": "Trait-Based",
    "NPI": "Narcissism-Based", "PNI": "Narcissism-Based", "FFNI": "Narcissism-Based",
    "FFNI_SF": "Narcissism-Based", "NARQ": "Narcissism-Based", "HSNS": "Narcissism-Based",
    "DTM": "Narcissism-Based", "DT4": "Narcissism-Based", "MCMIN": "Narcissism-Based",
    "IPN": "Narcissism-Based",
    "STBV": "Motivational", "MST": "Motivational", "RFT": "Motivational",
    "SDT": "Motivational", "AAM": "Motivational", "CS": "Motivational",
    "PCT": "Cognitive", "SCM": "Cognitive", "CEST": "Cognitive", "FSLS": "Cognitive",
    "MMPI": "Clinical", "TCI": "Clinical", "TMP": "Clinical", "BDI": "Clinical",
    "GAD7": "Clinical", "SCID": "Clinical", "MCMI": "Clinical",
    "RIT": "Clinical", "TAT": "Clinical", "WAIS": "Clinical",
    "TKI": "Interpersonal", "DISC": "Interpersonal",
    "RIASEC": "App/Holistic", "BT": "App/Holistic", "TEI": "App/Holistic",
    "EM": "App/Holistic", "PAPC": "App/Holistic", "CMOA": "App/Holistic",
}

for row in all_metadata:
    row['category'] = CATEGORY_MAP.get(row['model'], "Unknown")

df = pd.DataFrame(all_metadata)

print(f"\nMetadata:")
print(f"   Rows: {len(df):,}")
print(f"   Models: {df['model'].nunique()}")
print(f"   Categories: {df['category'].nunique()}")
print(f"   Unique factors: {df['factor'].nunique()}")
print(f"   Unique adjectives: {df['adjective'].nunique()}")
print(f"   Embedding dim: {X.shape[1]}")

## Section 3: Setup Colors & Imports

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Enable Colab plotly rendering
pio.renderers.default = 'colab'

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠️ umap-learn not installed. UMAP projections will be skipped.")

# Category colors (7 categories)
CATEGORY_COLORS = {
    "Trait-Based": "#1f77b4",
    "Narcissism-Based": "#9467bd",
    "Motivational": "#2ca02c",
    "Cognitive": "#ff7f0e",
    "Clinical": "#d62728",
    "Interpersonal": "#e377c2",
    "App/Holistic": "#7f7f7f",
}

# Model colors - 44 distinct colors
MODEL_COLORS = {
    "AAM": "#e377c2", "BDI": "#bcbd22", "BT": "#17becf", "CEST": "#1f77b4",
    "CMOA": "#ff7f0e", "CS": "#2ca02c", "DISC": "#d62728", "DT4": "#9467bd",
    "DTM": "#8c564b", "EM": "#e377c2", "EPM": "#7f7f7f", "FFNI": "#bcbd22",
    "FFNI_SF": "#17becf", "FTM": "#1f77b4", "FSLS": "#ff7f0e", "GAD7": "#2ca02c",
    "HEX": "#d62728", "HSNS": "#9467bd", "IPN": "#8c564b", "MBTI": "#e377c2",
    "MCMI": "#7f7f7f", "MCMIN": "#bcbd22", "MMPI": "#17becf", "MST": "#1f77b4",
    "NARQ": "#ff7f0e", "NPI": "#2ca02c", "OCEAN": "#d62728", "PAPC": "#9467bd",
    "PCT": "#8c564b", "PNI": "#e377c2", "RFT": "#7f7f7f", "RIASEC": "#bcbd22",
    "RIT": "#17becf", "SCID": "#1f77b4", "SCM": "#ff7f0e", "SDT": "#2ca02c",
    "SIXTEENPF": "#d62728", "STBV": "#9467bd", "TAT": "#8c564b", "TCI": "#e377c2",
    "TEI": "#7f7f7f", "TKI": "#bcbd22", "TMP": "#17becf", "WAIS": "#1f77b4",
}

print(f"✅ Imports and colors loaded")
print(f"✅ Plotly renderer: {pio.renderers.default}")
print(f"✅ Data summary: {X.shape[0]:,} embeddings × {X.shape[1]}-dim")
print(f"   Models: {df['model'].nunique()}, Categories: {df['category'].nunique()}")
print(f"   Memory: {X.nbytes / 1e6:.1f} MB")
print(f"\n📊 Category distribution:")
print(df['category'].value_counts().sort_values(ascending=False))

## Section 4: PCA Projection (2D + 3D)

In [ ]:
print("🔄 Computing PCA (50 components)...")
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X).astype(np.float32)

print(f"✅ PCA variance explained (50 components): {100*pca.explained_variance_ratio_.sum():.1f}%")
print(f"   PC1: {100*pca.explained_variance_ratio_[0]:.1f}%")
print(f"   PC2: {100*pca.explained_variance_ratio_[1]:.1f}%")
print(f"   PC3: {100*pca.explained_variance_ratio_[2]:.1f}%")

# 3D PCA visualization
fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter3d(
        x=X_pca[mask, 0],
        y=X_pca[mask, 1],
        z=X_pca[mask, 2],
        mode='markers',
        name=cat,
        marker=dict(
            size=3,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['model']}</b><br>Category: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Personality Atlas: PCA 3D Projection (6,694 factor chain embeddings)',
    scene=dict(
        xaxis_title=f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)",
        yaxis_title=f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)",
        zaxis_title=f"PC3 ({100*pca.explained_variance_ratio_[2]:.1f}%)",
    ),
    height=900, margin=dict(b=120),
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="top", y=-0.15, font=dict(size=11), xanchor="center", x=0.5, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show(renderer='colab')

print("\n💾 Saving PCA 3D figure...")
fig.write_html('pca_3d.html')
print("✅ Saved to pca_3d.html")

## Section 5: t-SNE Projection

In [ ]:
print("🔄 Computing t-SNE (PCA-50 pre-reduction for speed)...")
pca_50 = PCA(n_components=50)
X_pca_50 = pca_50.fit_transform(X).astype(np.float32)

tsne = TSNE(n_components=2, perplexity=25, learning_rate=10, random_state=42, max_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_pca_50).astype(np.float32)
print(f"✅ t-SNE computed (2D)")

# Compute category centroids
centroids = {}
for cat in df['category'].unique():
    mask = df['category'] == cat
    centroids[cat] = X_tsne[mask].mean(axis=0)

# t-SNE 2D visualization
fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0],
        y=X_tsne[mask, 1],
        mode='markers',
        name=cat,
        marker=dict(
            size=4,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['model']}</b><br>Category: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

# Add category centroids as stars
centroid_names = list(centroids.keys())
centroid_coords = np.array(list(centroids.values()))
fig.add_trace(go.Scatter(
    x=centroid_coords[:, 0],
    y=centroid_coords[:, 1],
    mode='markers+text',
    name='Category centroids',
    marker=dict(size=12, color='white', symbol='star', line=dict(width=2, color='black')),
    text=centroid_names,
    textposition='top center',
    hovertemplate='%{text}<extra></extra>',
    showlegend=True,
))

fig.update_layout(
    title='Personality Atlas: t-SNE 2D Projection (perplexity=25)',
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    height=900, margin=dict(b=120),
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="top", y=-0.15, font=dict(size=11), xanchor="center", x=0.5, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show(renderer='colab')

print("\n💾 Saving t-SNE figure...")
fig.write_html('tsne_2d.html')
print("✅ Saved to tsne_2d.html")

## Section 6: UMAP Projection (2D)

In [ ]:
from IPython.display import display, clear_output

print("🔄 Computing UMAP (PCA-50 pre-reduction for speed)...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, random_state=42, verbose=False)
X_umap = reducer.fit_transform(X_pca_50).astype(np.float32)
print(f"✅ UMAP computed (2D, neighbors=15)")

fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_umap[mask, 0],
        y=X_umap[mask, 1],
        mode='markers',
        name=cat,
        marker=dict(
            size=4,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['model']}</b><br>Category: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Personality Atlas: UMAP 2D Projection (neighbors=15)',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    height=900, margin=dict(b=120),
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="top", y=-0.15, font=dict(size=11), xanchor="center", x=0.5, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)

print("\n💾 Saving UMAP figure...")
fig.write_html('umap_2d.html')
print("✅ Saved to umap_2d.html")

display(fig)

## Section 7: Per-Category Cluster Analysis

In [ ]:
categories = sorted(df['category'].unique())
n_cats = len(categories)

fig = make_subplots(
    rows=(n_cats + 1) // 2,
    cols=2,
    subplot_titles=categories,
)

for idx, cat in enumerate(categories):
    row = (idx // 2) + 1
    col = (idx % 2) + 1
    mask = df['category'] == cat
    models = sorted(df[mask]['model'].unique())

    for model in models:
        model_mask = mask & (df['model'] == model)
        model_color = MODEL_COLORS.get(model, '#999999')

        fig.add_trace(
            go.Scatter(
                x=X_tsne[model_mask, 0],
                y=X_tsne[model_mask, 1],
                mode='markers',
                name=model,
                marker=dict(size=4, color=model_color, opacity=0.7, line=dict(width=0)),
                showlegend=(idx == 0),
                hovertemplate='<b>%{text}</b><extra></extra>',
                text=[model] * model_mask.sum(),
            ),
            row=row,
            col=col,
        )

fig.update_layout(
    title_text="Per-Category Clustering (t-SNE coordinates, colored by model)",
    height=1200,
    width=1400,
    template='plotly_dark',
    font=dict(size=11),
    showlegend=True,
    legend=dict(x=1.02, y=1.0, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show(renderer='colab')

print("\n💾 Saving per-category analysis...")
fig.write_html('per_category_tsne.html')
print("✅ Saved to per_category_tsne.html")

## Section 8: Category Centroid Similarity Heatmap

In [ ]:
print("🔄 Computing category centroid similarities...")

centroids_embed = {}
for cat in categories:
    mask = df['category'] == cat
    cat_embed = X[mask]
    centroid = cat_embed.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-10)
    centroids_embed[cat] = centroid

centroids_array = np.array(list(centroids_embed.values()))
sim_matrix = centroids_array @ centroids_array.T

fig = px.imshow(
    sim_matrix,
    x=categories,
    y=categories,
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    labels=dict(color='Cosine<br>Similarity'),
    title='Category Centroid Similarity Matrix (1536-Dim Embedding Space)',
    aspect='auto',
)

fig.update_traces(
    text=np.round(sim_matrix, 3),
    texttemplate='%{text}',
    textfont=dict(size=10),
)

fig.update_layout(
    height=700,
    width=750,
    template='plotly_dark',
    font=dict(size=11),
    xaxis_title='Category',
    yaxis_title='Category',
)
fig.show(renderer='colab')

print("\n💾 Saving heatmap...")
fig.write_html('category_similarity_heatmap.html')
print("✅ Saved to category_similarity_heatmap.html")

print("\n📊 Centroid Similarities (most similar pairs):")
similarities_list = []
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j:
            sim = sim_matrix[i, j]
            similarities_list.append((sim, cat1, cat2))

similarities_list.sort(reverse=True)
for sim, cat1, cat2 in similarities_list[:10]:
    print(f"  {cat1:20s} <-> {cat2:20s}: {sim:+.3f}")

## Section 9: Separation Metrics

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

print("🔄 Computing cluster separation and quality metrics...")

labels = df['category'].values
unique_cats = sorted(np.unique(labels))
label_map = {cat: i for i, cat in enumerate(unique_cats)}
label_ints = np.array([label_map[cat] for cat in labels])

# Global metrics (computed on full dataset)
silhouette = silhouette_score(X_pca_50, label_ints)
db_score = davies_bouldin_score(X_pca_50, label_ints)
ch_score = calinski_harabasz_score(X_pca_50, label_ints)

print("\n" + "="*60)
print("GLOBAL CLUSTER QUALITY METRICS")
print("="*60)
print(f"\n  Silhouette Score:        {silhouette:.4f}")
print("  Interpretation: -1 to 1; higher = better separated clusters")
print(f"  Rating: {'EXCELLENT' if silhouette > 0.5 else 'GOOD' if silhouette > 0.3 else 'MODERATE' if silhouette > 0.1 else 'WEAK'}")

print(f"\n  Davies-Bouldin Index:    {db_score:.4f}")
print("  Interpretation: Lower = better (0 = perfect separation)")
print(f"  Rating: {'EXCELLENT' if db_score < 1.0 else 'GOOD' if db_score < 2.0 else 'MODERATE'}")

print(f"\n  Calinski-Harabasz Score: {ch_score:.2f}")
print("  Interpretation: Higher = better cluster definition")

# Per-category silhouette: compute per-sample on full dataset, then average by category
print("\n" + "="*60)
print("PER-CATEGORY MEAN SILHOUETTE SCORES")
print("="*60)
sample_scores = silhouette_samples(X_pca_50, label_ints)
for cat in categories:
    mask = labels == cat
    n_samples = mask.sum()
    mean_score = sample_scores[mask].mean()
    print(f"  {cat:20s} (n={n_samples:4d}): {mean_score:+.4f}")

print("\n" + "="*60)
print("ATLAS COMPOSITION")
print("="*60)
print(f"  Total embeddings:          {len(df):,}")
print(f"  Embedding dimension:       {X.shape[1]} (OpenAI text-embedding-3-small)")
print(f"  Number of models:          {df['model'].nunique()}")
print(f"  Number of categories:      {len(categories)}")
print(f"  Mean embeddings per model: {len(df) / df['model'].nunique():.1f}")

print("\n" + "="*60)
print("EMBEDDINGS BY CATEGORY")
print("="*60)
for cat, count in df['category'].value_counts().sort_values(ascending=False).items():
    pct = 100 * count / len(df)
    print(f"  {cat:20s}: {count:4d} ({pct:5.1f}%)")

## Summary

**Generated Figures:**
- `pca_3d.html`, PCA 3D visualization (interactive)
- `tsne_2d.html`, t-SNE 2D global clustering with category centroids
- `umap_2d.html`, UMAP 2D projection
- `per_category_tsne.html`, Per-category cluster analysis colored by model
- `category_similarity_heatmap.html`, 7x7 centroid cosine similarity matrix

**Cluster Quality Metrics (Sections 8-9):**
- **Silhouette Score**: Measures how well-separated the 7 personality categories are in embedding space
- **Davies-Bouldin Index**: Quantifies cluster compactness and separation (lower = better)
- **Calinski-Harabasz Score**: Evaluates cluster definition quality (higher = better)

**Research Questions Addressed:**
- **RQ13 (Section 10):** Do the 7 disciplinary categories correspond to semantic boundaries? **No**, silhouette is near zero. Data-driven k=15 achieves 50x better separation.
- **RQ14 (Section 11):** What structure does the embedding space reveal? **15 SPA clusters** organized by meaning, not discipline. Clinical splits into 4, Narcissism into 3, Interpersonal Circumplex confirmed.

**The Semantic Personality Atlas (SPA):**
- The SPA reorganizes the atlas by meaning rather than disciplinary origin
- The embedding space has natural cluster structure that cuts across category boundaries
- Interpersonal is the only semantically distinct category
- Warmth and Dominance span all 7 disciplinary categories, confirming the Interpersonal Circumplex

**Atlas Details:**
- 44 personality models across 7 disciplinary categories
- 6,694 factor chain embeddings at 1536 dimensions (OpenAI text-embedding-3-small)
- Categories: Trait-Based, Narcissism-Based, Motivational, Cognitive, Clinical, Interpersonal, App/Holistic

**Paper Reference:** Raetano, J., Gregor, J., & Tamang, S. (2026). *A Survey and Computational Atlas of Personality Models.* ACM TIST.

## Section 10: RQ13, Embedding Space Topology (Semantic Personality Atlas)

**RQ13: Do the seven disciplinary categories correspond to semantic boundaries in the embedding space, or does the data organize around different structures?**

*(Paper 1, Section 6.14, "A Survey and Computational Atlas of Personality Models")*

The seven categories (Trait-Based, Narcissism-Based, Motivational, Cognitive, Clinical, Interpersonal, App/Holistic) were defined based on traditional disciplinary groupings within personality psychology. But the near-zero silhouette score (Section 9) suggests these boundaries may not correspond to natural semantic clusters. If data-driven clustering reveals a different structure, it would have implications for how personality models should be organized and compared.

**Approach:** Compare the disciplinary 7-category taxonomy against data-driven clustering (k-means with optimal k via elbow method and silhouette analysis) to determine whether the embedding space supports the current categorization or suggests a reorganization.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

print("=" * 60)
print("RQ: DISCIPLINARY vs DATA-DRIVEN CATEGORIZATION")
print("=" * 60)

# Step 1: Find optimal k using silhouette scores
print("\n[Step 1] Scanning k=2..15 for optimal cluster count...")
k_range = range(2, 16)
silhouette_scores = []
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_pca_50)
    sil = silhouette_score(X_pca_50, km_labels)
    silhouette_scores.append(sil)
    inertias.append(km.inertia_)
    print(f"  k={k:2d}: silhouette={sil:.4f}, inertia={km.inertia_:.0f}")

best_k = list(k_range)[np.argmax(silhouette_scores)]
best_sil = max(silhouette_scores)
print(f"\n  Best k={best_k} (silhouette={best_sil:.4f})")

# Step 2: Elbow + Silhouette plot
fig = make_subplots(rows=1, cols=2, subplot_titles=["Elbow Method (Inertia)", "Silhouette Score by k"])

fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers',
              name='Inertia', marker=dict(size=8)), row=1, col=1)
fig.add_trace(go.Scatter(x=list(k_range), y=silhouette_scores, mode='lines+markers',
              name='Silhouette', marker=dict(size=8, color='orange')), row=1, col=2)

# Mark best k
fig.add_trace(go.Scatter(x=[best_k], y=[best_sil], mode='markers',
              name=f'Best k={best_k}', marker=dict(size=14, color='red', symbol='star')),
              row=1, col=2)
# Mark k=7 (disciplinary)
sil_at_7 = silhouette_scores[5]  # index 5 = k=7
fig.add_trace(go.Scatter(x=[7], y=[sil_at_7], mode='markers',
              name=f'Disciplinary k=7', marker=dict(size=14, color='green', symbol='diamond')),
              row=1, col=2)

fig.update_layout(title='Optimal Cluster Count: Elbow + Silhouette Analysis',
                  height=500, template='plotly_dark', font=dict(size=12))
fig.update_xaxes(title_text='k (number of clusters)', row=1, col=1)
fig.update_xaxes(title_text='k (number of clusters)', row=1, col=2)
fig.update_yaxes(title_text='Inertia', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
display(fig)

In [ ]:
# Step 3: Compare disciplinary (7 categories) vs data-driven (best k) clustering
print("\n" + "=" * 60)
print(f"[Step 3] COMPARING: Disciplinary (k=7) vs Data-Driven (k={best_k})")
print("=" * 60)

# Disciplinary clustering
label_map_disc = {cat: i for i, cat in enumerate(sorted(df['category'].unique()))}
labels_disc = np.array([label_map_disc[c] for c in df['category']])
sil_disc = silhouette_score(X_pca_50, labels_disc)

# Data-driven clustering at best k
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_data = km_best.fit_predict(X_pca_50)
sil_data = silhouette_score(X_pca_50, labels_data)

# Also try k=7 data-driven for fair comparison
km_7 = KMeans(n_clusters=7, random_state=42, n_init=10)
labels_data_7 = km_7.fit_predict(X_pca_50)
sil_data_7 = silhouette_score(X_pca_50, labels_data_7)

# Agreement metrics
ari = adjusted_rand_score(labels_disc, labels_data_7)
nmi = normalized_mutual_info_score(labels_disc, labels_data_7)

print(f"\n  Disciplinary (7 categories):  silhouette = {sil_disc:.4f}")
print(f"  Data-driven (k=7):            silhouette = {sil_data_7:.4f}")
print(f"  Data-driven (k={best_k}):          silhouette = {sil_data:.4f}")
print(f"\n  Improvement (data k=7 vs disciplinary): {((sil_data_7 - sil_disc) / abs(sil_disc) * 100):+.1f}%")
print(f"  Improvement (data k={best_k} vs disciplinary): {((sil_data - sil_disc) / abs(sil_disc) * 100):+.1f}%")
print(f"\n  Agreement between disciplinary and data-driven (k=7):")
print(f"    Adjusted Rand Index: {ari:.4f} (1.0 = perfect agreement)")
print(f"    Normalized Mutual Info: {nmi:.4f} (1.0 = perfect agreement)")

# Step 4: What does data-driven clustering look like?
print("\n" + "=" * 60)
print(f"[Step 4] DATA-DRIVEN CLUSTER COMPOSITION (k={best_k})")
print("=" * 60)

for cluster_id in range(best_k):
    mask = labels_data == cluster_id
    n_in_cluster = mask.sum()
    cats_in_cluster = df[mask]['category'].value_counts()
    models_in_cluster = df[mask]['model'].nunique()
    dominant_cat = cats_in_cluster.index[0]
    dominant_pct = 100 * cats_in_cluster.iloc[0] / n_in_cluster
    
    print(f"\n  Cluster {cluster_id} (n={n_in_cluster}, {models_in_cluster} models):")
    print(f"    Dominant: {dominant_cat} ({dominant_pct:.0f}%)")
    for cat, count in cats_in_cluster.items():
        pct = 100 * count / n_in_cluster
        if pct >= 5:
            print(f"      {cat:20s}: {count:4d} ({pct:5.1f}%)")

In [ ]:
# Step 5: Visualize data-driven vs disciplinary on t-SNE
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Disciplinary (7 Categories)", f"Data-Driven (k={best_k})"])

# Left: disciplinary
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1],
        mode='markers', name=cat,
        marker=dict(size=3, color=CATEGORY_COLORS.get(cat, '#999'), opacity=0.6),
        showlegend=True,
    ), row=1, col=1)

# Right: data-driven
cluster_colors = px.colors.qualitative.Set1 + px.colors.qualitative.Set2
for c_id in range(best_k):
    mask = labels_data == c_id
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1],
        mode='markers', name=f'Cluster {c_id}',
        marker=dict(size=3, color=cluster_colors[c_id % len(cluster_colors)], opacity=0.6),
        showlegend=True,
    ), row=1, col=2)

fig.update_layout(
    title=f'Disciplinary Categories vs Data-Driven Clusters (k={best_k})',
    height=700, width=1400, template='plotly_dark', font=dict(size=11),
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5,
                font=dict(size=10), bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
    margin=dict(b=120),
)
display(fig)

In [ ]:
# Step 6: Answer the RQ
print("=" * 60)
print("RQ ANSWER: THE SEMANTIC PERSONALITY ATLAS (SPA)")
print("=" * 60)

print(f"""
FINDING: The seven atlas categories are DISCIPLINARY groupings,
not SEMANTIC groupings.

Evidence:
  1. Disciplinary categories produce near-zero cluster separation
     (silhouette = {sil_disc:.4f})

  2. Data-driven clustering at k=7 achieves {((sil_data_7 - sil_disc) / max(abs(sil_disc), 0.0001) * 100):+.0f}% better separation
     (silhouette = {sil_data_7:.4f})

  3. Optimal data-driven k={best_k} achieves silhouette = {sil_data:.4f}

  4. Agreement between disciplinary and data-driven (k=7):
     ARI = {ari:.4f}, NMI = {nmi:.4f}
     {"LOW agreement: data-driven clusters cut across category boundaries" if ari < 0.3 else "MODERATE agreement: some category structure preserved" if ari < 0.6 else "HIGH agreement: categories align with semantic clusters"}

INTERPRETATION:
  The categories reflect DISCIPLINARY ORIGINS (clinical psychology,
  trait theory, motivational science), not semantic distinctness.

  An adjective like "anxious" appears in Clinical (GAD-7, BDI),
  Trait-Based (OCEAN Neuroticism), and Narcissism-Based (vulnerable
  narcissism) models simultaneously. The embedding model correctly
  places these near each other because they share MEANING, even
  though they belong to different categories.

  Interpersonal models (TKI, DISC) are the exception because
  conflict-resolution vocabulary ("accommodating", "competing",
  "compromising") is semantically unique to that category.

IMPLICATION FOR PAPER 1:
  The Semantic Personality Atlas (SPA) reveals that the atlas
  categories are overlapping lenses on a shared semantic space,
  not cleanly separable dimensions. This is precisely WHY a
  unified atlas is needed: no single category covers the full space,
  and the boundaries between categories are empirically porous.
""")

## Section 11: RQ14, Semantic Reorganization (SPA Cluster Profiles)

**RQ14: What structure does the embedding space reveal when factor chains are clustered by meaning rather than disciplinary origin?**

*(Paper 1, Section 6.14, "A Survey and Computational Atlas of Personality Models")*

The data-driven clustering from RQ13 identified 15 optimal clusters that differ fundamentally from the 7 disciplinary categories. Below we label each cluster by its **dominant factor chain vocabulary** (factors and adjectives) to reveal what the embedding space actually organizes around: shared meaning, not disciplinary origin. Key findings: Clinical fragments into 4 clusters, Narcissism into 3, and Warmth and Dominance span all 7 categories, confirming the Interpersonal Circumplex as a cross-tradition universal.

In [ ]:
# Named semantic cluster labels (derived from dominant factor chain vocabulary)
SEMANTIC_CLUSTER_NAMES = {
    0: "Warmth & Prosociality",
    1: "Dominance & Persistence",
    2: "Clinical Psychopathology",
    3: "Depression & Anhedonia",
    4: "Activation & Approach",
    5: "Imagination & Openness",
    6: "Withdrawal & Avoidance",
    7: "Self-Direction & Autonomy",
    8: "Reactive Aggression",
    9: "Shame & Self-Punishment",
    10: "Dark Manipulation",
    11: "Grandiose Narcissism",
    12: "Anxiety & Fear",
    13: "Analytical Cognition",
    14: "Applied Life Domains",
}

print("=" * 70)
print(f"15 SPA CLUSTERS (Semantic Personality Atlas, data-driven, k={best_k})")
print("=" * 70)

cluster_profiles = []

for c_id in range(best_k):
    mask = labels_data == c_id
    cluster_df = df[mask]
    n = mask.sum()
    
    top_factors = cluster_df['factor'].value_counts().head(5)
    top_adjs = cluster_df['adjective'].value_counts().head(5)
    cat_breakdown = cluster_df['category'].value_counts()
    models = sorted(cluster_df['model'].unique())
    n_models = len(models)
    n_cats = len(cat_breakdown)
    
    semantic_label = SEMANTIC_CLUSTER_NAMES.get(c_id, f"Cluster {c_id}")
    
    cluster_profiles.append({
        'id': c_id, 'n': n, 'label': semantic_label,
        'n_models': n_models, 'n_cats': n_cats,
        'top_factors': top_factors, 'top_adjs': top_adjs,
        'cat_breakdown': cat_breakdown, 'models': models,
    })
    
    print(f"\n{'='*70}")
    print(f"  C{c_id}: {semantic_label} (n={n}, {n_models} models, {n_cats} categories)")
    print(f"{'='*70}")
    
    print(f"  Categories spanned:")
    for cat, count in cat_breakdown.items():
        pct = 100 * count / n
        bar = "█" * int(pct / 3)
        print(f"    {cat:20s}: {count:4d} ({pct:5.1f}%) {bar}")
    
    print(f"  Top factors:")
    for factor, count in top_factors.items():
        print(f"    {str(factor):35s}: {count:3d}")
    
    print(f"  Top adjectives:")
    for adj, count in top_adjs.items():
        print(f"    {str(adj):35s}: {count:3d}")

# Summary table
print("\n\n" + "=" * 70)
print("SUMMARY: 15 SPA CLUSTERS vs 7 DISCIPLINARY CATEGORIES")
print("=" * 70)
print(f"\n{'ID':>3s}  {'SPA Cluster Name':35s}  {'n':>5s}  {'Cats':>4s}  {'Dominant Category':20s}  {'%':>5s}")
print("-" * 85)
for p in cluster_profiles:
    dom_cat = p['cat_breakdown'].index[0]
    dom_pct = 100 * p['cat_breakdown'].iloc[0] / p['n']
    print(f"{p['id']:3d}  {p['label']:35s}  {p['n']:5d}  {p['n_cats']:4d}  {dom_cat:20s}  {dom_pct:5.1f}%")

# Key insights
multi_cat = sum(1 for p in cluster_profiles if p['n_cats'] >= 4)
pure_cat = sum(1 for p in cluster_profiles if 100 * p['cat_breakdown'].iloc[0] / p['n'] > 80)

print(f"\nKey Findings:")
print(f"  Clusters spanning 4+ categories: {multi_cat}/15 (shared constructs)")
print(f"  Category-pure clusters (>80%):   {pure_cat}/15 (domain-specific)")
print(f"  Clinical splits into: Psychopathology, Depression, Aggression, Anxiety")
print(f"  Narcissism splits into: Shame, Dark Manipulation, Grandiose")
print(f"  Warmth & Dominance span ALL 7 disciplinary categories")
print(f"\n  The embedding space organizes personality by MEANING, not by discipline.")

In [ ]:
# Visualize named semantic clusters on t-SNE
fig = go.Figure()

cluster_colors = px.colors.qualitative.Set1 + px.colors.qualitative.Set2 + px.colors.qualitative.Pastel1
for p in cluster_profiles:
    c_id = p['id']
    mask = labels_data == c_id
    color = cluster_colors[c_id % len(cluster_colors)]
    
    centroid = X_tsne[mask].mean(axis=0)
    
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0], y=X_tsne[mask, 1],
        mode='markers',
        name=f"C{c_id}: {p['label']}",
        marker=dict(size=4, color=color, opacity=0.6),
        text=[f"<b>C{c_id}: {p['label']}</b><br>Model: {row['model']}<br>"
              f"Factor: {row['factor']}<br>Adj: {row['adjective']}<br>Cat: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))
    
    fig.add_trace(go.Scatter(
        x=[centroid[0]], y=[centroid[1]],
        mode='text',
        text=[f"C{c_id}"],
        textfont=dict(size=10, color='white'),
        showlegend=False,
    ))

fig.update_layout(
    title=f'Personality Atlas: Semantic Personality Atlas (SPA): 15 Clusters by Trait Vocabulary',
    xaxis_title='t-SNE 1', yaxis_title='t-SNE 2',
    height=900, template='plotly_dark', font=dict(size=11),
    legend=dict(orientation="h", yanchor="top", y=-0.18, xanchor="center", x=0.5,
                font=dict(size=9), bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
    margin=dict(b=180),
)
display(fig)

print("\n💾 Saving semantic clusters figure...")
fig.write_html('spa_clusters_tsne.html')
print("✅ Saved to spa_clusters_tsne.html")